## What's next?

- Listen to your creations! 🎧
- Explore [Google AI Studio](https://aistudio.google.com/new_music) to iterate on your prompts visually.
- Check those cool AI Studio apps: [Lyria Studio](https://aistudio.google.com/apps/bundled/lyria_studio) where you can create songs wand generate karaoke versions of them and [Lyria rhythm](https://aistudio.google.com/apps/bundled/lyria_rhythm) that create custom song for a rhythm game.
- Combine music generation with [image generation](https://ai.google.dev/gemini-api/docs/image-generation) for full multimedia content.
- Dive into the [Lyria 3 technical report](https://deepmind.google/technologies/lyria/) to learn about the architecture behind these models.

In [3]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU bulunamadı. Runtime > Change runtime type > T4 GPU seçebilirsin.")

PyTorch version: 2.11.0+cpu
CUDA available: False
GPU bulunamadı. Runtime > Change runtime type > T4 GPU seçebilirsin.


In [4]:
!pip -q install jsbsim gymnasium numpy matplotlib pandas

In [7]:
import jsbsim
import os

In [8]:
import jsbsim
import os

fdm = jsbsim.FGFDMExec(None)
print("JSBSim version:", fdm.get_version())

root_dir = os.path.dirname(jsbsim.__file__)
print("JSBSim path:", root_dir)



     JSBSim Flight Dynamics Model v1.3.1 May 17 2026 14:26:04
            [JSBSim-ML v2.0]

JSBSim startup beginning ...

JSBSim version: 1.3.1 May 17 2026 14:26:04
JSBSim path: /usr/local/lib/python3.13/dist-packages/jsbsim


In [9]:
import os

aircraft_dir = os.path.join(root_dir, "aircraft")

print("Aircraft directory:")
print(aircraft_dir)

print("\nAircraft models:")

for name in os.listdir(aircraft_dir):
    print(name)

Aircraft directory:
/usr/local/lib/python3.13/dist-packages/jsbsim/aircraft

Aircraft models:
weather-balloon
fokker100
SGS
t6texan2
Submarine_Scout
c310
Boeing314
p51d
dr1
A4
blank
B747
B17
ballx
F80C
sgs233
J3Cub
c172x
c182
A320
f22
x24b
787-8
OV10
ah1s
J246
T37
F4N
Short_S23
c172p
MD11
f16
minisgs
pa28
sgs126
pc7
737
global5000
DHC6
f15
XB-70
aircraft_template.xml
L17
Camel
X15
L410
Shuttle
pogo-jsbsim
F450
wrightFlyer1903
T38
fokker50
Concorde
f104
ZLT-NT
Pterosaur
c172r
C130
paraglider
ball
mk82


In [10]:
import jsbsim
import os

fdm = jsbsim.FGFDMExec(root_dir)

fdm.set_aircraft_path(aircraft_dir)
fdm.load_model("f16")

fdm.set_dt(1.0 / 30.0)

print("F-16 loaded successfully.")



     JSBSim Flight Dynamics Model v1.3.1 May 17 2026 14:26:04
            [JSBSim-ML v2.0]

JSBSim startup beginning ...

Reading Aircraft Configuration File: General Dynamics F-16A
                            Version: 2.0


This aircraft model is a PRODUCTION release.

  Description:   Models an F-16A Block-32 (Basic US configuration)
  Model Author:  Erik Hofman
  Creation Date: 2001-12-28
  Version:       $Revision: 1.95 $

  Aircraft Metrics:
    WingArea: 300.000000
    WingSpan: 30.000000
    Incidence: 0.000000
    Chord: 11.320000
    H. Tail Area: 63.700000
    H. Tail Arm: 16.460000
    V. Tail Area: 54.750000
    V. Tail Arm: 0.000000
    Eyepoint (x, y, z): -336.200000 , 0.000000 , 29.500000
    Ref Pt (x, y, z): -189.500000 , 0.000000 , 3.900000
    Visual Ref Pt (x, y, z): -180.000000 , 0.000000 , 0.000000

  Mass and Balance:
    baseIxx: 9496.000000 slug-ft2
    baseIyy: 55814.000000 slug-ft2
    baseIzz: 63100.000000 slug-ft2
    baseIxy: -0.000000 slug-ft2
    baseI

In [12]:
# Initial conditions

fdm["ic/h-sl-ft"] = 10000.0
fdm["ic/vc-kts"] = 500.0

fdm["ic/psi-true-deg"] = 0.0
fdm["ic/phi-deg"] = 0.0
fdm["ic/theta-deg"] = 0.0

fdm["ic/alpha-deg"] = 2.0
fdm["ic/beta-deg"] = 0.0

fdm["ic/p-rad_sec"] = 0.0
fdm["ic/q-rad_sec"] = 0.0
fdm["ic/r-rad_sec"] = 0.0

fdm["ic/roc-fpm"] = 0.0

fdm["propulsion/engine[0]/set-running"] = -1

fdm.run_ic()

print("Initial conditions initialized.")


  Mass Properties Report (English units: lbf, in, slug-ft^2)
                                      Weight    CG-X    CG-Y    CG-Z         Ixx         Iyy         Izz         Ixy         Ixz         Iyz
    Base Vehicle                       17400.0  -193.0     0.0    -5.1      9496.0     55814.0     63100.0        -0.0      -982.0        -0.0
0   Pilot                                230.0  -336.2     0.0     0.0         0.0         0.0         0.0        -0.0         0.0        -0.0
0   Fuel                                  1500  -174.4      65       5           0           0           0
1   Fuel                                  1500  -174.4     -65       5           0           0           0
2   Fuel                                     0  -174.4      65     -15           0           0           0
3   Fuel                                     0  -174.4     -65     -15           0           0           0
                                                                                   

In [13]:
def get_state(fdm):

    state = {
        "altitude_ft": float(fdm["position/h-sl-ft"]),
        "airspeed_kts": float(fdm["velocities/vc-kts"]),

        "roll_deg": float(fdm["attitude/phi-deg"]),
        "pitch_deg": float(fdm["attitude/theta-deg"]),
        "heading_deg": float(fdm["attitude/psi-deg"]),

        "p": float(fdm["velocities/p-rad_sec"]),
        "q": float(fdm["velocities/q-rad_sec"]),
        "r": float(fdm["velocities/r-rad_sec"]),

        "alpha_deg": float(fdm["aero/alpha-deg"]),
        "beta_deg": float(fdm["aero/beta-deg"]),
    }

    return state


state = get_state(fdm)

for key, value in state.items():
    print(f"{key:20s}: {value:.4f}")

altitude_ft         : 10000.0000
airspeed_kts        : 500.0000
roll_deg            : 0.0000
pitch_deg           : 2.0000
heading_deg         : 0.0000
p                   : 0.0000
q                   : 0.0000
r                   : 0.0000
alpha_deg           : 2.0000
beta_deg            : 0.0000


In [14]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces

action_low = np.array([
    -1.0,   # aileron
    -1.0,   # elevator
    -1.0,   # rudder
     0.0    # throttle
], dtype=np.float32)

action_high = np.array([
     1.0,
     1.0,
     1.0,
     1.0
], dtype=np.float32)

action_space = spaces.Box(
    low=action_low,
    high=action_high,
    dtype=np.float32
)

print(action_space)

Box([-1. -1. -1.  0.], 1.0, (4,), float32)


In [15]:
def apply_action(fdm, action):

    aileron = float(np.clip(action[0], -1, 1))
    elevator = float(np.clip(action[1], -1, 1))
    rudder = float(np.clip(action[2], -1, 1))
    throttle = float(np.clip(action[3], 0, 1))

    fdm["fcs/aileron-cmd-norm"] = aileron
    fdm["fcs/elevator-cmd-norm"] = elevator
    fdm["fcs/rudder-cmd-norm"] = rudder
    fdm["fcs/throttle-cmd-norm"] = throttle

In [16]:
TARGET_ALTITUDE = 10000.0
TARGET_AIRSPEED = 500.0
TARGET_HEADING = 0.0

In [17]:
def angle_error_deg(target, current):

    error = target - current

    while error > 180:
        error -= 360

    while error < -180:
        error += 360

    return error


def get_observation(fdm):

    altitude = float(fdm["position/h-sl-ft"])
    airspeed = float(fdm["velocities/vc-kts"])

    roll = np.deg2rad(float(fdm["attitude/phi-deg"]))
    pitch = np.deg2rad(float(fdm["attitude/theta-deg"]))
    heading = float(fdm["attitude/psi-deg"])

    p = float(fdm["velocities/p-rad_sec"])
    q = float(fdm["velocities/q-rad_sec"])
    r = float(fdm["velocities/r-rad_sec"])

    alpha = np.deg2rad(float(fdm["aero/alpha-deg"]))
    beta = np.deg2rad(float(fdm["aero/beta-deg"]))

    altitude_error = (TARGET_ALTITUDE - altitude) / 10000.0
    airspeed_error = (TARGET_AIRSPEED - airspeed) / 500.0

    heading_error = angle_error_deg(
        TARGET_HEADING,
        heading
    ) / 180.0

    obs = np.array([
        altitude_error,
        airspeed_error,
        heading_error,

        np.sin(roll),
        np.cos(roll),

        np.sin(pitch),
        np.cos(pitch),

        p,
        q,
        r,

        np.sin(alpha),
        np.cos(alpha),

        np.sin(beta),
        np.cos(beta),

    ], dtype=np.float32)

    return obs

In [18]:
observation_space = spaces.Box(
    low=-np.inf,
    high=np.inf,
    shape=(14,),
    dtype=np.float32
)

print(observation_space)

Box(-inf, inf, (14,), float32)


In [19]:
def compute_reward(fdm):

    altitude = float(fdm["position/h-sl-ft"])
    airspeed = float(fdm["velocities/vc-kts"])
    heading = float(fdm["attitude/psi-deg"])

    altitude_error = abs(TARGET_ALTITUDE - altitude)
    airspeed_error = abs(TARGET_AIRSPEED - airspeed)

    heading_error = abs(
        angle_error_deg(TARGET_HEADING, heading)
    )

    altitude_penalty = altitude_error / 1000.0
    airspeed_penalty = airspeed_error / 50.0
    heading_penalty = heading_error / 30.0

    reward = -(
        altitude_penalty +
        airspeed_penalty +
        heading_penalty
    )

    return float(reward)

In [20]:
class F16Env(gym.Env):

    metadata = {"render_modes": []}

    def __init__(self):

        super().__init__()

        self.dt = 1.0 / 30.0

        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(14,),
            dtype=np.float32
        )

        self.action_space = spaces.Box(
            low=np.array(
                [-1, -1, -1, 0],
                dtype=np.float32
            ),
            high=np.array(
                [1, 1, 1, 1],
                dtype=np.float32
            ),
            dtype=np.float32
        )

        self.fdm = None
        self.step_count = 0
        self.max_steps = 3000

    def create_fdm(self):

        fdm = jsbsim.FGFDMExec(root_dir)

        fdm.set_aircraft_path(aircraft_dir)
        fdm.load_model("f16")

        fdm.set_dt(self.dt)

        fdm["ic/h-sl-ft"] = TARGET_ALTITUDE
        fdm["ic/vc-kts"] = TARGET_AIRSPEED

        fdm["ic/psi-true-deg"] = TARGET_HEADING
        fdm["ic/phi-deg"] = 0.0
        fdm["ic/theta-deg"] = 2.0

        fdm["ic/alpha-deg"] = 2.0
        fdm["ic/beta-deg"] = 0.0

        fdm["ic/p-rad_sec"] = 0.0
        fdm["ic/q-rad_sec"] = 0.0
        fdm["ic/r-rad_sec"] = 0.0

        fdm.run_ic()

        return fdm

    def reset(self, seed=None, options=None):

        super().reset(seed=seed)

        self.fdm = self.create_fdm()
        self.step_count = 0

        obs = get_observation(self.fdm)

        return obs, {}

    def step(self, action):

        apply_action(self.fdm, action)

        self.fdm.run()

        self.step_count += 1

        obs = get_observation(self.fdm)

        reward = compute_reward(self.fdm)

        altitude = float(
            self.fdm["position/h-sl-ft"]
        )

        airspeed = float(
            self.fdm["velocities/vc-kts"]
        )

        terminated = False

        if altitude < 1000:
            terminated = True

        if altitude > 50000:
            terminated = True

        if airspeed < 100:
            terminated = True

        if airspeed > 1000:
            terminated = True

        truncated = self.step_count >= self.max_steps

        info = {
            "altitude": altitude,
            "airspeed": airspeed,
            "heading": float(
                self.fdm["attitude/psi-deg"]
            )
        }

        return obs, reward, terminated, truncated, info

In [21]:
env = F16Env()

obs, info = env.reset()

print("Observation shape:", obs.shape)
print("Observation:")
print(obs)

print("\nAction space:")
print(env.action_space)



     JSBSim Flight Dynamics Model v1.3.1 May 17 2026 14:26:04
            [JSBSim-ML v2.0]

JSBSim startup beginning ...

Reading Aircraft Configuration File: General Dynamics F-16A
                            Version: 2.0


This aircraft model is a PRODUCTION release.

  Description:   Models an F-16A Block-32 (Basic US configuration)
  Model Author:  Erik Hofman
  Creation Date: 2001-12-28
  Version:       $Revision: 1.95 $

  Aircraft Metrics:
    WingArea: 300.000000
    WingSpan: 30.000000
    Incidence: 0.000000
    Chord: 11.320000
    H. Tail Area: 63.700000
    H. Tail Arm: 16.460000
    V. Tail Area: 54.750000
    V. Tail Arm: 0.000000
    Eyepoint (x, y, z): -336.200000 , 0.000000 , 29.500000
    Ref Pt (x, y, z): -189.500000 , 0.000000 , 3.900000
    Visual Ref Pt (x, y, z): -180.000000 , 0.000000 , 0.000000

  Mass and Balance:
    baseIxx: 9496.000000 slug-ft2
    baseIyy: 55814.000000 slug-ft2
    baseIzz: 63100.000000 slug-ft2
    baseIxy: -0.000000 slug-ft2
    baseI